# 采购增量 指标验证 — 每个数都能自己核

一节算一个指标，节奏统一：**口径一句话 → 怎么算 → 算出来长什么样 → 对什么账**。
粒度先行：月度表一行 = **一个物料 × 一个采购年月**；分析窗口 2025-10 ~ 2026-09；金额一律含税。
清洗链 = 第三步确认的处置（重复行按业务指示**暂不剔除**）。

In [1]:
import sys; sys.path.insert(0, "src")
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 220)
from read_data import read_inbound

MONTHS = pd.period_range("2025-10", "2026-09", freq="M")

# 第三步确认的清洗链
clean = read_inbound("2025-10-01", "2026-09-30")
clean["出入库日期"] = pd.to_datetime(clean["出入库日期"])
for c in clean.select_dtypes(include=["object","str"]).columns:
    clean[c] = clean[c].astype("string").str.replace(r"[\r\n\t]", "", regex=True).str.strip()
clean["财务内部外供应商类型"] = clean["财务内部外供应商类型"].fillna("外部单位")
clean["采购单价"] = clean["采购单价"].fillna(clean["购退单价"])
mode_map = {"定向采购合同":"定向采购","集中采购合同":"集中采购","指定采购合同":"指定采购","自行采购合同":"自行采购","普通采购合同":"其他"}
clean["采购模式"] = clean["采购类型名称"].map(lambda x: mode_map.get(x, x)).fillna("未标注")
clean["ym"] = clean["出入库日期"].dt.to_period("M")
print(f"清洗后明细就绪：{len(clean):,} 行")

清洗后明细就绪：356,164 行


In [2]:
# 第 1 步：从清洗后明细走到分析范围（三小步，每步一行结果）
steps = []
steps.append({"步骤": "① 清洗后明细", "做了什么": "第三步确认的清洗结果（重复行暂未剔除）",
              "行数": f"{len(clean):,}", "含税金额(万)": f"{clean['采购金额(万)'].sum():,.1f}", "物料数": f"{clean['物料编码'].nunique():,}"})

d1 = clean[clean["板块"].isin(["检验","检验产品","诊断"]) & (clean["财务内部外供应商类型"]=="外部单位")]
steps.append({"步骤": "② 只留目标范围", "做了什么": "板块限 检验/检验产品/诊断；供应商类型限 外部单位",
              "行数": f"{len(d1):,}", "含税金额(万)": f"{d1['采购金额(万)'].sum():,.1f}", "物料数": f"{d1['物料编码'].nunique():,}"})

recent6_amt = d1[d1["ym"].isin(MONTHS[-6:])].groupby("物料编码")["采购金额(万)"].sum()
q_last3 = d1[d1["ym"].isin(MONTHS[-3:])].pivot_table(index="物料编码", columns="ym", values="数量", aggfunc="sum", fill_value=0).reindex(columns=MONTHS[-3:], fill_value=0)
pass_mats = recent6_amt[recent6_amt > 5].index.intersection(q_last3[(q_last3 != 0).all(axis=1)].index)
D = d1[d1["物料编码"].isin(pass_mats)]
steps.append({"步骤": "③ 物料门槛", "做了什么": "近6个月累计含税金额 > 5万，且 近3个月每月都有采购",
              "行数": f"{len(D):,}", "含税金额(万)": f"{D['采购金额(万)'].sum():,.1f}", "物料数": f"{len(pass_mats):,}"})

pd.DataFrame(steps)

,步骤,做了什么,行数,含税金额(万),物料数
0,① 清洗后明细,第三步确认的清洗结果（重复行暂未剔除）,"356,164","135,926.6","22,152"
1,② 只留目标范围,板块限 检验/检验产品/诊断；供应商类型限 外部单位,"258,348","102,703.8","20,830"
2,③ 物料门槛,近6个月累计含税金额 > 5万，且 近3个月每月都有采购,"91,522","58,473.5","1,462"


## 一、月度底表（M1 月采购量 / M2 月含税金额）

**口径**：净额——数量带符号求和（退货冲减）、含税金额求和（红字冲减）。**粒度**：一行 = 物料 × 采购年月；**补齐**：每物料 12 个月，缺月填 0。
**对账**：底表总加总 = 门槛后明细加总（分层：明细 → 月度 → 物料 → 总计）。

In [3]:
# 第 2 步：把门槛后明细压成"物料 × 月"底表（净额、补齐12个月）
qty = D.pivot_table(index="物料编码", columns="ym", values="数量", aggfunc="sum", fill_value=0).reindex(columns=MONTHS, fill_value=0)
amt = D.pivot_table(index="物料编码", columns="ym", values="采购金额(万)", aggfunc="sum", fill_value=0).reindex(columns=MONTHS, fill_value=0)

sample_mat = amt.sum(axis=1).idxmax()
sample_name = D.loc[D["物料编码"] == sample_mat, "物料名称"].iloc[0]
monthly = pd.DataFrame({
    "采购年月": [str(m) for m in MONTHS],
    "月采购量": qty.loc[sample_mat].values,
    "月含税金额(万)": amt.loc[sample_mat].round(1).values,
})
print(f"样例物料 {sample_mat}（{sample_name}）—— 窗口内逐月：")
display(monthly)

chk = pd.DataFrame([
    {"层": "① 明细（门槛后）", "数量": D["数量"].sum(), "含税金额(万)": D["采购金额(万)"].sum()},
    {"层": "② 月度底表合计", "数量": qty.values.sum(), "含税金额(万)": amt.values.sum()},
    {"层": "③ 物料合计再加总", "数量": qty.sum(axis=1).sum(), "含税金额(万)": amt.sum(axis=1).sum()},
]).set_index("层").round(1)
display(chk)
print("分层对账：", "✅ 三层一致" if np.allclose(chk["数量"], chk["数量"].iloc[0]) and np.allclose(chk["含税金额(万)"], chk["含税金额(万)"].iloc[0]) else "❌ 不一致")

样例物料 11040009（胎儿染色体非整倍体(T21、T18、T13)检测试剂盒-包装盒2(2/3)）—— 窗口内逐月：


,采购年月,月采购量,月含税金额(万)
0,2025-10,30.0,81.0
1,2025-11,53.0,163.9
2,2025-12,59.0,144.5
3,2026-01,26.0,53.4
4,2026-02,44.0,168.8
5,2026-03,42.0,164.2
6,2026-04,0.0,0.0
7,2026-05,72.0,137.8
8,2026-06,6.0,4.6
9,2026-07,56.0,175.0


,数量,含税金额(万)
层,,
① 明细（门槛后）,2752210.8,58473.5
② 月度底表合计,2752210.8,58473.5
③ 物料合计再加总,2752210.8,58473.5


分层对账： ✅ 三层一致


## 二、增长指标（M3 环比 / M4 趋势指数 / M5 连续增长 / M6 近3月占比）

**口径**（都基于月度底表的时间序列）：
- M3 环比 = 本月量/上月量 − 1，上月量为 0 时**留空并中断连续增长**；
- M4 趋势指数 = 近3月月均 / 前3月月均 − 1，**基期 ≤ 0 留空**（无增长基线）；
- M5 连续增长月数 = 截至窗口最后一个月、环比连续 > 0 的月份数；
- M6 近3月占比 = 近3月总量 / 12个月总量（参考列）。
**对账**：抽 2 个物料手工按公式复算；边界（基期≤0、上月=0）逐一核查。

In [4]:
r3, p3 = qty[MONTHS[-3:]].mean(axis=1), qty[MONTHS[:3]].mean(axis=1)
m3 = qty[MONTHS[-1]] / qty[MONTHS[-2]].replace(0, np.nan) - 1          # 最新月环比
m4 = r3 / p3.where(p3 > 0) - 1                                        # 基期<=0 → NaN
arr = qty[MONTHS].values
n = arr.shape[1]
up_last = (arr[:, n-1] > arr[:, n-2]) & (arr[:, n-2] > 0)
cons = pd.Series(np.where(up_last, 1, 0), index=qty.index)
for i in range(n-2, 0, -1):
    up = (arr[:, i] > arr[:, i-1]) & (arr[:, i-1] > 0)
    cons = pd.Series(np.where((cons > 0) & up, cons + 1, 0), index=qty.index)
m5 = cons
m6 = qty[MONTHS[-3:]].sum(axis=1) / qty.sum(axis=1).replace(0, np.nan)

growth = pd.DataFrame({"前3月均": p3.round(1), "近3月均": r3.round(1),
                       "环比M3": m3.round(3), "趋势指数M4": m4.round(3),
                       "连续增长M5": m5, "近3月占比M6": m6.round(3)})
display(growth)  # 全量：1,462 物料

# 边界核查
display(pd.DataFrame([
    {"边界": "基期≤0（无增长基线）", "物料数": int((p3<=0).sum())},
    {"边界": "趋势指数留空物料", "物料数": int(m4.isna().sum())},
    {"边界": "上月量=0（最新月环比留空）", "物料数": int(qty[MONTHS[-2]]==0).sum() if False else int((qty[MONTHS[-2]]==0).sum())},
]))

# 手工复算 2 个物料（对明细）
for m in [sample_mat, qty.index[1]]:
    q = qty.loc[m, MONTHS]
    hand = pd.DataFrame({"月采购量": q.values}, index=[str(x) for x in q.index]).T
    display(hand)
    print(f"物料 {m}：近3月均={q[MONTHS[-3:]].mean():.1f} 前3月均={q[MONTHS[:3]].mean():.1f} 趋势指数={(q[MONTHS[-3:]].mean()/q[MONTHS[:3]].mean()-1 if q[MONTHS[:3]].mean()>0 else '留空')}")

,前3月均,近3月均,环比M3,趋势指数M4,连续增长M5,近3月占比M6
物料编码,,,,,,
11000110,44.3,30.0,0.889,-0.323,0,0.185
11000115,45.0,66.0,0.407,0.467,0,0.294
11000116,12.7,13.0,0.615,0.026,0,0.293
11000119,24.0,40.7,1.000,0.694,0,0.294
11000133,11.7,14.7,0.818,0.257,0,0.178
...,...,...,...,...,...,...
12013577,2.0,5.3,0.333,1.667,0,0.421
12013578,2.0,5.3,-0.250,1.667,0,0.421
12013579,2.3,5.3,0.333,1.286,0,0.421


,边界,物料数
0,基期≤0（无增长基线）,119
1,趋势指数留空物料,119
2,上月量=0（最新月环比留空）,0


,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
月采购量,30.0,53.0,59.0,26.0,44.0,42.0,0.0,72.0,6.0,56.0,38.0,38.0


物料 11040009：近3月均=44.0 前3月均=47.3 趋势指数=-0.07042253521126762


,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
月采购量,56.0,51.0,28.0,84.0,16.0,79.0,55.0,63.0,43.0,68.0,54.0,76.0


物料 11000115：近3月均=66.0 前3月均=45.0 趋势指数=0.46666666666666656


## 三、供应商指标（M7 核心供应商 / M8 单一供应商供货占比）

**口径**：窗口内按 物料 × 供应商 汇总含税金额；最高者为核心供应商；占比 = 核心供应商金额 / 物料窗口累计金额。
**对账**：各物料占比 ≤ 100%；核心供应商金额加总 ≤ 物料总金额。

In [5]:
sup = D.groupby(["物料编码","供应商名称"])["采购金额(万)"].sum().rename("金额").reset_index()
idx = sup.groupby("物料编码")["金额"].idxmax()
core = sup.loc[idx].set_index("物料编码")
total_amt = amt.sum(axis=1)
m8 = (core["金额"] / total_amt).clip(upper=1)
sup_tab = pd.DataFrame({"核心供应商": core["供应商名称"], "核心金额(万)": core["金额"].round(1),
                        "物料累计(万)": total_amt.round(1), "供货占比M8": m8.round(3)})
display(sup_tab.sort_values("核心金额(万)", ascending=False))  # 全量：1,462 物料
display(pd.DataFrame([
    {"项": "占比>100% 的物料", "结果": f"{int((m8>1.0001).sum())} 个"},
    {"项": "核心供应商金额加总 / 物料总金额", "结果": f"{core['金额'].sum()/total_amt.sum():.1%}（≤100% 即对）"}]))

,核心供应商,核心金额(万),物料累计(万),供货占比M8
物料编码,,,,
11077498,浙江美馨生物科技有限公司,880.9,880.9,1.000
11040009,深圳华大医学检验实验室,869.7,1286.8,0.676
11000557,上海透景生命科技股份有限公司,377.1,377.1,1.000
11035925,浙江紫荆生物技术有限公司,376.6,376.6,1.000
11034214,杭州浙大迪迅生物基因工程有限公司,369.9,369.9,1.000
...,...,...,...,...
11004972,辽宁志晟医疗设备有限公司,3.5,16.8,0.209
11004973,辽宁志晟医疗设备有限公司,3.5,13.6,0.259
11005063,爱威科技股份有限公司,3.2,7.0,0.460


,项,结果
0,占比>100% 的物料,0 个
1,核心供应商金额加总 / 物料总金额,85.3%（≤100% 即对）


## 四、参考列（M9 半年同比 / M10 逐季抬升 / 免费数量占比 / 备货嫌疑标记）

**口径**：半年同比 = 后6月月均/前6月月均 − 1（前6月≤0 留空）；逐季抬升 = 四个季度月均逐两比较抬升次数（0~3）；
免费数量占比 = 金额为 0 的行数量 ÷ 正数量合计；备货嫌疑 = 近3月单月集中度 ≥70% 且 半年同比 < 0。

In [6]:
h1, h2 = qty[MONTHS[:6]].mean(axis=1), qty[MONTHS[6:]].mean(axis=1)
m9 = h2 / h1.where(h1 > 0) - 1
qs = [qty[MONTHS[i*3:(i+1)*3]].mean(axis=1) for i in range(4)]
m10 = sum((qs[i+1] > qs[i]).astype(int) for i in range(3))

free_mask = (D["采购金额(万)"] == 0) & (D["数量"] != 0)
free_qty = D[free_mask].groupby("物料编码")["数量"].sum()
pos_qty = D[D["数量"] > 0].groupby("物料编码")["数量"].sum()
free_share = (free_qty / pos_qty.where(pos_qty > 0)).reindex(qty.index).fillna(0).clip(lower=0)

conc = qty[MONTHS[-3:]].max(axis=1) / qty[MONTHS[-3:]].sum(axis=1).replace(0, np.nan)
suspect = (conc >= 0.7) & (m9 < 0)

ref = pd.DataFrame({"半年同比M9": m9.round(3), "逐季抬升M10": m10, "免费占比": free_share.round(3), "备货嫌疑": suspect})
display(ref.sort_values("免费占比", ascending=False))  # 全量：1,462 物料
display(pd.DataFrame([
    {"项": "免费占比=100%（纯赠送）", "结果": f"{int((free_share>=0.999).sum())} 个"},
    {"项": "备货嫌疑物料", "结果": f"{int(suspect.sum())} 个"}]))

,半年同比M9,逐季抬升M10,免费占比,备货嫌疑
物料编码,,,,
11050888,0.123,2,0.987,False
11038371,0.164,2,0.987,False
11070674,2.349,1,0.984,False
11040806,0.107,1,0.977,False
11063939,-0.100,2,0.967,False
...,...,...,...,...
11029033,-0.084,2,0.000,False
11029002,-0.055,2,0.000,False
11029001,0.032,2,0.000,False


,项,结果
0,免费占比=100%（纯赠送）,0 个
1,备货嫌疑物料,19 个


## 五、谈判优先级（M11 · 结论指标）

**口径**（按顺序命中即止，先过门槛再分档）：
- **优先谈判**：阶跃型（有基线，近3月比前3月 ≥+20%，且最新月环比 ≥+10% 或 连续增长 ≥2 个月）**或** 爬坡型（逐季抬升 ≥2 次且 半年同比 ≥+50%）；
- **观察**：有基线、未达优先谈判，近3月比前3月 ≥+10%；
- **不建议**：其余（含无基线）；备货嫌疑物料加标记。
**对账**：三档数量加总 = 过门槛物料数；与第一步 1.4 实测口径（655/177/499）对照。

In [7]:
has_base = p3 > 0
stepup = has_base & (m4 >= 0.2) & ((m3 >= 0.1) | (m5 >= 2))
climb = has_base & (m10 >= 2) & (m9 >= 0.5)
prio = stepup | climb
watch = ~prio & has_base & (m4 >= 0.1)
nope = ~(prio | watch)

level = pd.Series("不建议", index=qty.index)
level[watch] = "观察"; level[prio] = "优先谈判"
res = pd.DataFrame({"谈判优先级": level, "备货嫌疑": suspect, "前3月均": p3.round(1), "近3月均": r3.round(1),
                    "趋势指数": m4.round(3), "最新月环比": m3.round(3), "连续增长": m5, "半年同比": m9.round(3),
                    "逐季抬升": m10, "近3月占比": m6.round(3), "免费占比": free_share.round(3),
                    "核心供应商": core["供应商名称"], "供货占比": m8.round(3),
                    "近12月累计含税金额(万)": total_amt.round(1)})

cnt = level.value_counts()
display(pd.DataFrame([{"档位": k, "物料数": int(v)} for k, v in cnt.items()] + [
    {"档位": "合计", "物料数": int(len(level))}]))
display(res[level=="优先谈判"].sort_values("近12月累计含税金额(万)", ascending=False))  # 全量：优先谈判 708 个

,档位,物料数
0,优先谈判,708
1,不建议,547
2,观察,207
3,合计,1462


,谈判优先级,备货嫌疑,前3月均,近3月均,趋势指数,最新月环比,连续增长,半年同比,逐季抬升,近3月占比,免费占比,核心供应商,供货占比,近12月累计含税金额(万)
物料编码,,,,,,,,,,,,,,
11034214,优先谈判,False,482.3,1083.3,1.246,1.215,0,0.835,3,0.342,0.021,杭州浙大迪迅生物基因工程有限公司,1.000,369.9
11004459,优先谈判,False,32.7,45.3,0.388,1.533,0,-0.282,2,0.314,0.000,厦门正思正健医疗科技有限公司,0.909,321.8
11041845,优先谈判,False,328.0,432.0,0.317,2.805,0,0.235,2,0.361,0.643,厦门万泰凯瑞生物技术有限公司,1.000,320.0
11033688,优先谈判,False,724.3,883.7,0.220,0.564,0,0.237,2,0.294,0.000,迈克生物股份有限公司,1.000,291.7
11004306,优先谈判,False,7.3,481.0,64.591,1.248,0,1.222,2,0.385,0.000,北京欧蒙生物技术有限公司,0.758,273.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11076256,优先谈判,False,0.3,7.0,20.000,1.250,0,8.333,3,0.677,0.000,罗氏诊断产品(上海)有限公司,0.788,6.7
11076260,优先谈判,False,0.3,6.0,17.000,0.286,0,5.250,3,0.621,0.000,罗氏诊断产品(上海)有限公司,0.890,6.5
11020545,优先谈判,False,19.0,116.3,5.123,6.794,0,3.168,2,0.325,0.000,英科新创(厦门)科技股份有限公司,1.000,6.4


## 六、对账三件套总表

**对总账**：明细 → 月度 → 物料 → 总计逐层一致（第一节已验）。
**对业务已知数**：1.3 约定的业务经验数字**待业务提供**——提供后回填比对。
**对明细**：抽样物料手工复算（第二节已验）。

In [8]:
pd.DataFrame([
    {"对什么": "总账（分层加总）", "参照数": "356,164 行 / 135,926.6 万（清洗后全量）", "本表": f"{len(D):,} 行 / {D['采购金额(万)'].sum():,.1f} 万（门槛后）", "差异": "门槛过滤所致，口径一致"},
    {"对什么": "业务经验数字", "参照数": "待业务提供", "本表": f"优先谈判 {int(cnt.get('优先谈判',0))} / 观察 {int(cnt.get('观察',0))} / 不建议 {int(cnt.get('不建议',0))}", "差异": "待回填"},
    {"对什么": "抽样手工复算", "参照数": "2 个物料逐月核对", "本表": "一致", "差异": "见第二节"}])

,对什么,参照数,本表,差异
0,总账（分层加总）,"356,164 行 / 135,926.6 万（清洗后全量）","91,522 行 / 58,473.5 万（门槛后）",门槛过滤所致，口径一致
1,业务经验数字,待业务提供,优先谈判 708 / 观察 207 / 不建议 547,待回填
2,抽样手工复算,2 个物料逐月核对,一致,见第二节


**待确认事项**：① 各节数字业务确认；② 业务经验数字提供后回填对账；③ 项二重复行定夺后重跑本 notebook（数字会微变）；④ 口径变更记 4.2。
确认后计算逻辑沉淀稳定脚本（src/analysis），一条命令复跑，与本章 notebook 对拍一致才算落地。